## Run chemistry and synthesis locally


## A native state kernel is reused along a classical geometry sweep

The visible qfunc prepares the one-bit tapered reference and applies its
variational Pauli evolution. FlagQuantum compiles this declared state program.
Host chemistry code rebuilds a Hamiltonian for each of twelve distances and
classically optimizes its angle, starting from zero at every geometry.

Try the nonzero preview binding before changing the distance grid. The
500-iteration ceiling is a per-geometry optimizer limit, not a fixed number
of successful iterations or a compiler search budget. Exact local expectation,
finite active-space model error and variational optimization error differ.
Propagate ansatz edits to shared main.py for the full curve; a low energy
curve does not alone establish physical accuracy outside the chosen basis.


In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import qfunc
from flagquantum.synth import suzuki_trotter

from examples.synthesis.applications.chemistry.molecular_energy_curve.main import (
    DEFAULT_PARAMS,
    GENERATOR,
    prepare_reference,
)


@qfunc
def native_main(params: list[float], state: Output[QArray[QBit]]) -> None:
    prepare_reference(state)
    suzuki_trotter(GENERATOR, params[0], 1, 1, state)


def build_preview(params: tuple[float, ...] = DEFAULT_PARAMS) -> synth.Program:
    if len(params) != 1:
        raise ValueError("The retained molecular ansatz requires one parameter")
    return synth.synthesize(
        native_main, classical_inputs={"params": params}, max_workspace=0
    )


from IPython.display import Image, display

from examples.synthesis.applications.chemistry.molecular_energy_curve.overview.main import (
    report,
    save_figure,
)
from examples.synthesis.applications.chemistry.molecular_energy_curve.overview.model import (
    distances,
)
from examples.synthesis.applications.chemistry.molecular_energy_curve.overview.workflow import (
    ITERATION_LIMIT,
    run_workflow,
)


## Keep all twelve original distances


In [ ]:
grid = distances()
assert len(grid) == 12
print("Bond distances in angstroms", grid)


## Set the local variational iteration limit


In [ ]:
assert ITERATION_LIMIT == 500
print("Run every molecule with zero initial variational angle.")


## Rebuild and execute a VQE model at each distance


In [ ]:
points = run_workflow()
report(points)


## Plot the binding curve


In [ ]:
display(Image(filename=str(save_figure(points))))


preview = build_preview((0.37,))
print("Editable native resources:", preview.report["resources"])
